# Building a Synthetic Image Dataset with Sana

_Authored by: [Parag Ekbote](https://github.com/ParagEkbote)_ and [David Berenstein](https://github.com/davidberenstein1957)

In [ ]:
!pip install -q pruna metaflow datasets

In [ ]:
from metaflow import FlowSpec, step
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText
from diffusers import SanaPipeline
from pruna import smash, SmashConfig
import torch
from PIL import Image
from pathlib import Path
import random

class SyntheticImageFlow(FlowSpec):

    @step
    def start(self):
        print("📥 Loading a subset of prompt-image preference dataset...")
        dataset = load_dataset("data-is-better-together/open-image-preferences-v1", split="train[:1%]")
        self.pairs = [
            (ex["prompt"], ex["preferred"]) for ex in dataset
            if ex["preferred"] in [0, 1]
        ]
        self.next(self.select_prompts)

    @step
    def select_prompts(self):
        print("🔍 Selecting prompts from preferred image pairs...")
        self.selected_prompts = [prompt for prompt, _ in self.pairs]
        print(f"✅ {len(self.selected_prompts)} prompts selected.")
        self.next(self.rewrite_prompts)

    @step
    def rewrite_prompts(self):
        print("✍️ Rewriting prompts for variety...")
        adjectives = ["surreal", "epic", "dreamy", "vibrant", "macro", "eerie"]
        self.rewritten_prompts = [
            f"{random.choice(adjectives)} {prompt}" for prompt in self.selected_prompts
        ]
        self.next(self.generate_images)

    @step
    def generate_images(self):
        print("🖼️ Generating images using SanaPipeline + Pruna's smash()...")

        # Load and optimize the pipeline
        pipe = SanaPipeline.from_pretrained(
            "Efficient-Large-Model/Sana_600M_512px_diffusers",
            torch_dtype=torch.float16
        ).to("cuda")

        smashed_pipe = smash(
            model=pipe,
            smash_config=SmashConfig(compilers=["diffusers2"]),
        )

        self.generated = []
        output_dir = Path("outputs")
        output_dir.mkdir(exist_ok=True)

        for i, prompt in enumerate(self.rewritten_prompts[:50]):  # Keep it small for testing
            image = smashed_pipe(prompt).images[0]
            image_path = output_dir / f"image_{i}.png"
            image.save(image_path)
            self.generated.append((prompt, str(image_path)))

        self.next(self.evaluate_images)

    @step
    def evaluate_images(self):
        print("🧠 Scoring images with BLIP-2...")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = AutoProcessor.from_pretrained.from_pretrained("google/gemma-3-4b-it")
    model = AutoModelForImageTextToText.from_pretrained(
        "google/gemma-3-4b-it", torch_dtype=torch.float16
    ).to(device)

    def score_prompt_image(prompt, image_path):
        image = Image.open(image_path).convert("RGB")
        # Construct VQA prompt: "Does this image match: <prompt>?"
        question = f"Does this image match the prompt: '{prompt}'? Answer yes or no."
        inputs = processor(images=image, text=question, return_tensors="pt").to(device)
        output = model.generate(**inputs)
        answer = processor.tokenizer.decode(output[0], skip_special_tokens=True).lower()
        return 1.0 if "yes" in answer else 0.0

    self.scored_images = []
    for prompt, img_path in self.generated:
        score = score_prompt_image(prompt, img_path)
        self.scored_images.append((prompt, img_path, score))

    self.scored_images.sort(key=lambda x: x[2], reverse=True)
    self.next(self.end)

    @step
    def end(self):
        print("🍾 Done! Top 5 generated images by CLIP score:")
        for i, (prompt, path, score) in enumerate(self.scored_images[:5]):
            print(f"{i+1}. {prompt} → {path} [score: {score:.3f}]")

if __name__ == "__main__":
    SyntheticImageFlow()
